Initial setup for the application
- Creates databases
- creates volumes
- creates tables below and populates them with initial data
  - employees
  - robots
  - shelves
  - bins

In [0]:
%pip install faker

In [0]:
%pip install faker faker-commerce

In [0]:
%sql
create database if not exists amazon_fullfilment_bronze;
create database if not exists amazon_fullfilment_silver;
create database if not exists amazon_fullfilment_gold;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.amazon_fullfilment_bronze.landing_zone;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.amazon_fullfilment_silver.processing_zone;

In [0]:
import uuid
from pyspark.sql import functions as F
from datetime import datetime

catalog_name = "workspace"
database_name = "amazon_fullfilment_bronze"

bronze_volume_path ="/Volumes/workspace/amazon_fullfilment_bronze/landing_zone/"
silver_volume_path ="/Volumes/workspace/amazon_fullfilment_silver/processing_zone/"
source_volume_path = "/Volumes/workspace/default/amazon_fullfilment/source/"

# Generate a unique ID for this specific notebook run
#current_run_uuid = str(uuid.uuid4())
# 93f7758a-f5a2-4b6f-bf40-da5b67878c02
current_run_uuid = "2"

def add_bronze_metadata(df):
    """
    Standardizes the metadata for all Bronze tables.
    """
    return df.withColumn("_ingest_timestamp", F.current_timestamp()) \
             .withColumn("_source_file", F.col("_metadata.file_path")) \
             .withColumn("_batch_id", F.lit(current_run_uuid))

In [0]:
# Insert into Bronze layer function
 
def ingest_to_bronze(table_name, schema, source_path, checkpoint_path):
    """
    Standardizes the Auto Loader ingestion for any table.
    """
    raw_stream = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .schema(schema)
        .load(source_path))
    
    # We select * and _metadata to ensure the global function works
    return (add_bronze_metadata(raw_stream.select("*", "_metadata"))
        .writeStream
        .option("checkpointLocation", f"{checkpoint_path}/_checkpoint")
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(f"{catalog_name}.{database_name}.{table_name}"))

# Creating the Bronze layer tables

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_bronze.orders (
    order_id STRING NOT NULL,
    customer_id STRING NOT NULL,
    orderdate DATE,
    status STRING NOT NULL,
    updated_timestamp TIMESTAMP NOT NULL,
    _batch_id STRING,    
    _ingest_timestamp TIMESTAMP,  -- When Spark processed it
    _source_file STRING           -- Which CSV it came from (for debugging)
)
CLUSTER BY (status, order_id)
TBLPROPERTIES (delta.autoOptimize.optimizeWrite = true);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_bronze.order_items (
    order_id STRING NOT NULL,
    order_item_id STRING NOT NULL,
    product_id STRING NOT NULL,
    quantity INTEGER,
    _batch_id STRING,
    _ingest_timestamp TIMESTAMP NOT NULL,  -- When Spark processed it
    _source_file STRING           -- Which CSV it came from (for debugging)
)
CLUSTER BY (order_id)
TBLPROPERTIES (delta.autoOptimize.optimizeWrite = true);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_bronze.shelves_events (
    shelf_id STRING NOT NULL,
    robot_id STRING NOT NULL,
    max_weight_capacity DOUBLE,
    status STRING NOT NULL,
    updated_timestamp TIMESTAMP NOT NULL,
    _batch_id STRING,
    _ingest_timestamp TIMESTAMP,  -- When Spark processed it
    _source_file STRING           -- Which CSV it came from (for debugging)
)
USING DELTA
CLUSTER BY (shelf_id, status)
TBLPROPERTIES (delta.autoOptimize.optimizeWrite = true);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_bronze.inventory (
    shelf_id STRING NOT NULL,
    product_id STRING NOT NULL,
    quantity INTEGER,
    order_id STRING NOT NULL,
    _batch_id STRING,
    _ingest_timestamp TIMESTAMP,  -- When Spark processed it
    _source_file STRING           -- Which CSV it came from (for debugging)
)
USING DELTA
CLUSTER BY (shelf_id)
TBLPROPERTIES (delta.autoOptimize.optimizeWrite = true);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_bronze.robot_events (
    robot_id STRING NOT NULL,
    status STRING NOT NULL,
    max_weight_capacity DOUBLE,
    battery_level INTEGER,
    event_timestamp TIMESTAMP,
    _batch_id STRING,
    _ingest_timestamp TIMESTAMP,  -- When Spark processed it
    _source_file STRING           -- Which CSV it came from (for debugging)
)
USING DELTA
CLUSTER BY (status)
TBLPROPERTIES (delta.autoOptimize.optimizeWrite = true);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_bronze.bin_events (
    bin_id STRING NOT NULL,
    order_id STRING,
    item_count INTEGER,
    status STRING NOT NULL,
    current_weight DOUBLE,
    employee_id STRING,
    updated_timestamp TIMESTAMP NOT NULL,
    _batch_id STRING,
    _ingest_timestamp TIMESTAMP,  -- When Spark processed it
    _source_file STRING           -- Which CSV it came from (for debugging)
)
USING DELTA
CLUSTER BY (bin_id)
TBLPROPERTIES (delta.autoOptimize.optimizeWrite = true);

# Creating Silver layer tables

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_silver.customer_silver (
  customer_id STRING NOT NULL,
  first_name STRING,
  last_name STRING,
  gender STRING,                
  email STRING,
  phone STRING,
  updated_timestamp TIMESTAMP
) 
TBLPROPERTIES (delta.enableChangeDataFeed = true);
ALTER TABLE workspace.amazon_fullfilment_silver.customer_silver ALTER COLUMN customer_id SET NOT NULL;
Alter table workspace.amazon_fullfilment_silver.customer_silver  add CONSTRAINT customers_pk PRIMARY KEY(customer_id);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_silver.address_silver (
  Address_ID STRING NOT NULL,
  Customer_ID STRING,
  Street_Address STRING,
  City STRING NOT NULL,      
  Province STRING NOT NULL,  
  Postal_Code STRING,
  Updated_Timestamp TIMESTAMP
) 
TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_silver.employee (
  employee_sk STRING NOT NULL,         -- Surrogate Key (usually a hash of id + start_date)
  employee_id STRING NOT NULL,
  first_name STRING NOT NULL,
  last_name STRING NOT NULL,
  job_role STRING NOT NULL,      
  certification_level STRING,
  current_station_id STRING,
  shift_id STRING,
  is_active BOOLEAN,
  start_date TIMESTAMP,
  end_date TIMESTAMP,
  Updated_Timestamp TIMESTAMP)
USING DELTA
CLUSTER BY (employee_id)
TBLPROPERTIES (
  delta.enableChangeDataFeed = true,
  delta.columnMapping.mode = 'name'
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_silver.products_silver (
  product_id STRING NOT NULL,
  sku STRING,
  product_name STRING NOT NULL,
  category STRING NOT NULL,
  weight_kg DOUBLE,
  is_current BOOLEAN,
  start_date TIMESTAMP,
  end_date TIMESTAMP,
 Updated_Timestamp TIMESTAMP)
 USING DELTA
 TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_silver.orders_silver (
  order_sk STRING NOT NULL,         -- Surrogate Key (usually a hash of id + start_date)
  order_id STRING NOT NULL,         -- Natural Key
  customer_id STRING NOT NULL,
  orderdate DATE,
  status STRING,
  updated_timestamp TIMESTAMP,
  is_current BOOLEAN NOT NULL,      -- Flag for the latest record
  start_date TIMESTAMP NOT NULL,    -- Beginning of this status version
  end_date TIMESTAMP,               -- Null if current, or timestamp of status change
  
  -- Constraints for Modeling Excellence
  CONSTRAINT orders_pk PRIMARY KEY(order_sk),
  CONSTRAINT orders_customer_fk FOREIGN KEY(customer_id) 
    REFERENCES workspace.amazon_fullfilment_silver.customer_silver(customer_id)
 )
 USING DELTA
 TBLPROPERTIES (
   delta.enableChangeDataFeed = true,
   delta.columnMapping.mode = 'name'
 )

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_silver.order_items_silver (
  order_item_sk STRING NOT NULL,   -- Hash of order_item_id
  order_item_id STRING NOT NULL,   -- Natural Key from source
  order_id STRING NOT NULL,         -- FK to Orders
  product_id STRING NOT NULL,       -- FK to Products
  quantity INTEGER,
  unit_price DECIMAL(10,2),        -- Good practice to include price at time of order
  updated_timestamp TIMESTAMP,
  
  -- Constraints
  CONSTRAINT order_items_pk PRIMARY KEY(order_item_sk)
 )
 USING DELTA
 CLUSTER BY (order_id, product_id) 
 TBLPROPERTIES (
   delta.enableChangeDataFeed = true
 );

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_silver.inventory_silver (
  inventory_sk STRING NOT NULL,   -- Hash of inventory_id
  shelf_id STRING NOT NULL,   -- Natural Key from source
  product_id STRING NOT NULL,       -- FK to Products
  quantity INTEGER,
  order_id STRING,
  updated_timestamp TIMESTAMP,
  
  -- Constraints
  CONSTRAINT inventory_pk PRIMARY KEY(inventory_sk)
 )
 USING DELTA
 CLUSTER BY (shelf_id, product_id) 
 TBLPROPERTIES (
   delta.enableChangeDataFeed = true
 );

In [0]:
%sql

CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_silver.robots (
  robot_sk STRING NOT NULL,        -- Surrogate Key
  robot_id STRING NOT NULL,        -- Primary Key
  robot_type STRING,               -- 'Heavy_Lifter', 'Sorter', 'Picker'

  -- Hardware Specifications (Static)
  max_weight_capacity DOUBLE,      -- To prevent "Out of Balance" robot picking
  max_speed_mps DOUBLE,            -- Meters per second for efficiency logic
  
  -- Current Operational State (Dynamic)
  status STRING,                   -- 'Idle', 'Active', 'Charging', 'Maintenance', 'Error'
  current_battery_pct INT,      -- Critical for "Return to Base" logic
  last_maintenance_date DATE,
  
  is_active BOOLEAN,
  start_date TIMESTAMP,
  end_date TIMESTAMP,
  Updated_Timestamp TIMESTAMP)

USING DELTA
CLUSTER BY (robot_id) 
TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_silver.shelves (
  shelf_sk STRING NOT NULL,        -- Surrogate Key
  shelf_id STRING NOT NULL,        -- Primary Key

  -- Physical Characteristics
  robot_id STRING,         -- Link back to the robot (if being carried)
  max_weight_capacity DOUBLE,         
  status STRING,                   -- 'In_Storage', 'In_Transit', 'Being_Restocked'

  is_active BOOLEAN,
  start_date TIMESTAMP,
  end_date TIMESTAMP,
  Updated_Timestamp TIMESTAMP)

USING DELTA
CLUSTER BY (shelf_id) 
TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_silver.bins (
  bins_sk STRING NOT NULL,        -- Surrogate Key
  bin_id STRING NOT NULL,          -- Unique identifier (often a barcode)

  -- Order Context
  order_id STRING,         -- The specific order this bin is collecting for
  item_count INT,                  -- Number of items currently in the bin
  -- Workflow State
  status STRING,               -- 'Empty', 'Collecting', 'Full_Waiting_Pack', 'At_Packing', 'Labeled'
  current_weight DOUBLE,
  
  -- Audit
  employee_id STRING,         -- ID of the last person to touch the bin
  is_active BOOLEAN,
  start_date TIMESTAMP,
  end_date TIMESTAMP,
  updated_timestamp TIMESTAMP
) 
USING DELTA
CLUSTER BY (bin_id)
TBLPROPERTIES (delta.enableChangeDataFeed = true);

# Populate the following tables for the warehouse operations:
- employee
- robots
- shelves
- inventory
- bins 


In [0]:
# Generate employee data

from pyspark.sql import Row
from faker import Faker
import random
from pyspark.sql.functions import current_timestamp

# Initialize Faker with Canadian English locale
fake = Faker('en_CA')

def generate_employee_bundle(num_employees=100):
    employees = []
    
    # Define roles and their probabilities
    # Picker, Packer, Stower get ~31.6% each, Supervisor gets 5%
    roles = ['Picker', 'Packer', 'Stower', 'Supervisor']
    weights = [0.316, 0.316, 0.316, 0.052] 

    for _ in range(num_employees):
        emp_id = f"VHN-{fake.unique.random_int(min=1000, max=9999)}"
        
        # 1. Assign role based on probability
        role = random.choices(roles, weights=weights, k=1)[0]
        
        # 2. Apply conditional logic for Supervisor
        if role == 'Supervisor':
            cert_level = None  # Empty per requirements
            # Higher pay bracket for supervisors: 40 - 55
            pay_rate = round(random.uniform(40.0, 55.0), 2)
        else:
            cert_level = random.choice(['L1', 'L2', 'L3'])
            # Standard bracket: 25 - 40
            pay_rate = round(random.uniform(25.0, 40.0), 2)

        employees.append(Row(
            employee_id=emp_id,
            first_name=fake.first_name(),
            last_name=fake.last_name(),
            job_role=role,
            certification_level=cert_level,
            current_station_id="",
            shift_id=random.choice(['Morning', 'Afternoon', 'Overnight']),
         #   is_active=True,
            hourly_rate_target=pay_rate,
            updated_timestamp="" # Will be populated by Spark function
        ))
            
    # Convert to DataFrame and add the timestamp via Spark for consistency
    return spark.createDataFrame(employees).withColumn("updated_timestamp", current_timestamp())

# Create the initial batch
df_initial_employees = generate_employee_bundle(50)
#display(df_initial_employees)

#save to the source volume
(df_initial_employees.write
.format("csv")
.option("header",True)
.option("delimiter",",")
.mode("append")
.save(f"{source_volume_path}/employee"))

#df_initial_employees.write.format("delta").mode("append").saveAsTable("workspace.amazon_fullfilment_bronze.employees")

In [0]:
# Insert into Bronze layer employee table
 
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, DateType, IntegerType, BooleanType

# Define exactly what the data should look like
employee_schema = StructType([
    StructField("employee_id", StringType(), False),
    StructField("first_name", StringType(), False),
    StructField("last_name", StringType(), False),
    StructField("job_role", StringType(), False),
    StructField("certification_level", StringType(), True),
    StructField("current_station_id", StringType(), True),
    StructField("shift_id", StringType(), False),
    StructField("hourly_rate_target", DoubleType(), True),
    StructField("updated_timestamp", TimestampType(), False)
])

# 1. Define your paths
bronze_employee_path = f"{bronze_volume_path}employee"

ingest_to_bronze("employee", employee_schema, f"{source_volume_path}employee", bronze_employee_path)

In [0]:
dbutils.fs.ls("/Volumes/workspace/default/amazon_fullfilment/source")

In [0]:
dbutils.fs.ls("/Volumes/workspace/default/amazon_fullfilment/source/robot_events/")

In [0]:
# process new records into the Silver layer employee table
from pyspark.sql import functions as F

def process_scd2_employee(microBatchDF, batchId):
    # 1. Get a list of employee_id that already exist in Silver
    # This prevents duplicating brand new orders
    target_table = "workspace.amazon_fullfilment_silver.employee"
    
    # 2. Split the micro-batch logic
    # Part A: The 'Insert' half (Always NULL merge_key)
    inserts_df = microBatchDF.withColumn("merge_key", F.lit(None))
    
    # Part B: The 'Update' half (Only for IDs that already exist in Silver)
    # This prevents the 'double insert' on the first run
    existing_ids = microBatchDF.sparkSession.table(target_table) \
        .filter("is_active = true") \
        .select("employee_id")
    
    updates_df = microBatchDF.join(existing_ids, "employee_id", "inner") \
        .withColumn("merge_key", F.col("employee_id"))
    
    # 3. Combine and Merge
    combined_df = inserts_df.unionByName(updates_df, allowMissingColumns=True)
    combined_df.createOrReplaceTempView("source_employee_view")

    microBatchDF.sparkSession.sql(f"""
        MERGE INTO {target_table} AS target
        USING source_employee_view AS source
        ON target.employee_id = source.merge_key AND target.is_active = true
        
        -- If we find the ID and status changed, expire the old one
        WHEN MATCHED AND target.job_role <> source.job_role AND target.shift_id <> source.shift_id THEN
          UPDATE SET target.is_active = false, target.end_date = source.updated_timestamp, target.updated_timestamp = source.updated_timestamp
          
        -- If it's the 'NULL' half of the union, it will always be NOT MATCHED -> Insert
        WHEN NOT MATCHED THEN
          INSERT (employee_sk, employee_id, first_name, last_name, job_role, certification_level, current_station_id, shift_id, is_active, start_date, end_date, updated_timestamp)
          VALUES (md5(concat(source.employee_id, cast(source.updated_timestamp as STRING))), 
                  source.employee_id, source.first_name, source.last_name, source.job_role, source.certification_level, source.current_station_id, source.shift_id, true, source.updated_timestamp, NULL, source.updated_timestamp)
    """)

    # 3. Start the Stream
query = (spark.readStream
    .table("workspace.amazon_fullfilment_bronze.employee")
    .filter("employee_id IS NOT NULL") # Basic data quality
    .writeStream
    .foreachBatch(process_scd2_employee)
    .option("checkpointLocation", f"{silver_volume_path}/checkpoints/silver_employee_scd2")
    .trigger(availableNow=True) # Or .trigger(processingTime='1 minute') for 24/7
    .start())

In [0]:
#Generate initial robots data

from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType, DateType
from pyspark.sql.functions import current_timestamp, to_date
from faker import Faker
from datetime import datetime 
from pyspark.sql import Row
import random

# Define the schema to match your Silver table exactly
robot_schema = StructType([
    StructField("robot_id", StringType(), False),
    StructField("status", StringType(), False),
    StructField("max_weight_capacity", DoubleType(), True),
    StructField("battery_level", IntegerType(), True),    
    StructField("event_timestamp", TimestampType(), True)
])

def generate_robot_batch(num_records=1000):
    fake = Faker()
    robot_ids = [f"ROB_{i:03d}" for i in range(1, 1001)]
    
    data = []
    for rid in robot_ids:
        now = datetime.now()
        record = Row(
            robot_id=rid,
            status="Idle",
            max_weight_capacity=500.0,
            battery_level=int(random.randint(15, 100)), # Python int
            event_timestamp=now
        )
        data.append(record)
    
    # Apply the schema here
    return spark.createDataFrame(data, schema=robot_schema)

# Run and Write
df_initial_robots = generate_robot_batch(1000)
#display(df_initial_robots)
#df_initial_robots.write.format("delta").mode("overwrite").saveAsTable("workspace.amazon_fullfilment_silver.robots")

#save to the source volume
(df_initial_robots.write
.format("csv")
.option("header",True)
.option("delimiter",",")
.mode("append")
.save(f"{source_volume_path}/robot_events"))

In [0]:
# Insert into Bronze layer robot_events table
 
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, DateType, IntegerType

# Define exactly what the data should look like
robot_events_schema = StructType([
    StructField("robot_id", StringType(), True),
    StructField("status", StringType(), True),
    StructField("max_weight_capacity", DoubleType(), True),
    StructField("battery_level", IntegerType(), True),
    StructField("event_timestamp", TimestampType(), True)
])

# 1. Define your paths
bronze_robot_events_path = f"{bronze_volume_path}robots"

ingest_to_bronze("robot_events", robot_events_schema, f"{source_volume_path}robot_events", bronze_robot_events_path)

In [0]:
# process new records into the Silver layer robots table
from pyspark.sql import functions as F
from pyspark.sql.functions import current_timestamp, current_date

def process_scd2_robots(microBatchDF, batchId):
    # 1. Get a list of order_ids that already exist in Silver
    # This prevents duplicating brand new orders
    target_table = "workspace.amazon_fullfilment_silver.robots"
    
    # 2. Split the micro-batch logic
    # Part A: The 'Insert' half (Always NULL merge_key)
    inserts_df = microBatchDF.withColumn("merge_key", F.lit(None))
    
    # Part B: The 'Update' half (Only for IDs that already exist in Silver)
    # This prevents the 'double insert' on the first run
    existing_ids = microBatchDF.sparkSession.table(target_table) \
        .filter("is_active = true") \
        .select("robot_id")
    
    updates_df = microBatchDF.join(existing_ids, "robot_id", "inner") \
        .withColumn("merge_key", F.col("robot_id"))
    
    # 3. Combine and Merge
    combined_df = inserts_df.unionByName(updates_df, allowMissingColumns=True)
    combined_df.createOrReplaceTempView("source_robot_view")

    microBatchDF.sparkSession.sql(f"""
        MERGE INTO {target_table} AS target
        USING source_robot_view AS source
        ON target.robot_id = source.merge_key AND target.is_active = true
        
        -- If we find the ID and status changed, expire the old one
        WHEN MATCHED AND target.status <> source.status THEN
          UPDATE SET target.is_active = false, target.end_date = source.event_timestamp, target.updated_timestamp = source.event_timestamp
          
        -- If it's the 'NULL' half of the union, it will always be NOT MATCHED -> Insert
        WHEN NOT MATCHED THEN
          INSERT (robot_sk, robot_id, robot_type, max_weight_capacity, max_speed_mps, status, current_battery_pct, last_maintenance_date,  is_active, start_date, end_date, updated_timestamp)
          VALUES (md5(concat(source.robot_id, cast(source.event_timestamp as STRING))), 
                  source.robot_id, 'Hercules', source.max_weight_capacity, 10.0, source.status, source.battery_level, current_date(), 
                  true, source.event_timestamp, NULL, source.event_timestamp)
    """)

    # 3. Start the Stream
query = (spark.readStream
    .table("workspace.amazon_fullfilment_bronze.robot_events")
    .filter("status IS NOT NULL") # Basic data quality
    .writeStream
    .foreachBatch(process_scd2_robots)
    .option("checkpointLocation", f"{silver_volume_path}/checkpoints/silver_robots_scd2")
    .trigger(availableNow=True) # Or .trigger(processingTime='1 minute') for 24/7
    .start())

In [0]:
#Generate Shelves

import uuid
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

# 1. Get the list of existing Robot IDs
robots_list = [row['robot_id'] for row in spark.read.table("workspace.amazon_fullfilment_silver.robots").select("robot_id").distinct().collect()]

# 2. Define the base shelf generation logic
def generate_shelves_spark(num_shelves=30000):
    # Create a range dataframe to start with
    df = spark.range(num_shelves)
    
    # Define a UDF for UUID if you want them unique per row
    uuid_udf = F.udf(lambda: str(uuid.uuid4()), StringType())
    
    # Add the static columns
    df = df.withColumn("shelf_id", uuid_udf()) \
           .withColumn("max_weight_capacity", F.lit(30.00).cast(DoubleType())) \
           .withColumn("status", F.lit("Idle")) \
           .withColumn("updated_timestamp", F.current_timestamp())
    
    # 3. Assign Random Robot IDs
    # Create a mapping of robots to an index
    robots_df = spark.createDataFrame([(r,) for r in robots_list], ["robot_id"])
    robots_df = robots_df.withColumn("robot_idx", F.monotonically_increasing_id())
    num_robots = len(robots_list)
    
    # Assign a random index to each shelf and join
    df = df.withColumn("rand_idx", (F.rand() * num_robots).cast("long"))
    
    final_df = df.join(robots_df, df.rand_idx == robots_df.robot_idx, "left") \
                 .select("shelf_id", "robot_id", "max_weight_capacity", "status", "updated_timestamp")
    
    return final_df

# 4. Generate the 30,000 records
shelves_df = generate_shelves_spark(30000)
#display(shelves_df)
# 5. Save to the source volume as CSV
(shelves_df.write
    .format("csv")
    .option("header", True)
    .option("delimiter", ",")
    .mode("overwrite") # Use overwrite if you are 'cleaning up'
    .save(f"{source_volume_path}/shelves_events"))

print(f"Successfully generated 30,000 shelves assigned to {len(robots_list)} unique robots.")

In [0]:
# Insert into Bronze layer Shelves table
 
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, DateType, IntegerType

# Define exactly what the data should look like
shelve_schema = StructType([
    StructField("shelf_id", StringType(), True),
    StructField("robot_id", StringType(), True),
    StructField("max_weight_capacity", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("updated_timestamp", TimestampType(), True)
])

# 1. Define your paths
bronze_shelves_path = f"{bronze_volume_path}shelves_events"

ingest_to_bronze("shelves_events", shelve_schema, f"{source_volume_path}shelves_events", bronze_shelves_path)

In [0]:
# process new records into the Silver layer shelves table
from pyspark.sql import functions as F
from pyspark.sql.functions import current_timestamp, current_date

def process_scd2_shelves(microBatchDF, batchId):
    # 1. Get a list of order_ids that already exist in Silver
    # This prevents duplicating brand new orders
    target_table = "workspace.amazon_fullfilment_silver.shelves"
    
    # 2. Split the micro-batch logic
    # Part A: The 'Insert' half (Always NULL merge_key)
    inserts_df = microBatchDF.withColumn("merge_key", F.lit(None))
    
    # Part B: The 'Update' half (Only for IDs that already exist in Silver)
    # This prevents the 'double insert' on the first run
    existing_ids = microBatchDF.sparkSession.table(target_table) \
        .filter("is_active = true") \
        .select("shelf_id")
    
    updates_df = microBatchDF.join(existing_ids, "shelf_id", "inner") \
        .withColumn("merge_key", F.col("shelf_id"))
    
    # 3. Combine and Merge
    combined_df = inserts_df.unionByName(updates_df, allowMissingColumns=True)
    combined_df.createOrReplaceTempView("source_shelves_view")

    microBatchDF.sparkSession.sql(f"""
        MERGE INTO {target_table} AS target
        USING source_shelves_view AS source
        ON target.shelf_id = source.merge_key AND target.is_active = true
        
        -- If we find the ID and status changed, expire the old one
        WHEN MATCHED AND target.status <> source.status THEN
          UPDATE SET target.is_active = false, target.end_date = source.updated_timestamp, target.updated_timestamp = source.updated_timestamp
          
        -- If it's the 'NULL' half of the union, it will always be NOT MATCHED -> Insert
        WHEN NOT MATCHED THEN
          INSERT (shelf_sk, shelf_id, robot_id, max_weight_capacity, status, is_active, start_date, end_date, updated_timestamp)
          VALUES (md5(concat(source.shelf_id, cast(source.updated_timestamp as STRING))), 
                  source.shelf_id, source.robot_id, source.max_weight_capacity, source.status,  
                  true, source.updated_timestamp, NULL, source.updated_timestamp)
    """)

    # 3. Start the Stream
query = (spark.readStream
    .table("workspace.amazon_fullfilment_bronze.shelves_events")
    .filter("status IS NOT NULL") # Basic data quality
    .writeStream
    .foreachBatch(process_scd2_shelves)
    .option("checkpointLocation", f"{silver_volume_path}/checkpoints/silver_shelves_scd2")
    .trigger(availableNow=True) # Or .trigger(processingTime='1 minute') for 24/7
    .start())

In [0]:
# process new records into the Silver layer order_items table
from pyspark.sql import functions as F

def process_scd1_order_items(microBatchDF, batchId):
    # 1. Get a list of order_ids that already exist in Silver
    # This prevents duplicating brand new orders
    target_table = "workspace.amazon_fullfilment_silver.order_items_silver"
    
    # 2. Split the micro-batch logic
    # Part A: The 'Insert' half (Always NULL merge_key)
    inserts_df = microBatchDF.withColumn("merge_key", F.lit(None))
    
    # Part B: The 'Update' half (Only for IDs that already exist in Silver)
    # This prevents the 'double insert' on the first run
    existing_ids = microBatchDF.sparkSession.table(target_table).select("order_item_id")
       # .filter("is_current = true") \
    
    
    updates_df = microBatchDF.join(existing_ids, "order_item_id", "inner") \
        .withColumn("merge_key", F.col("order_item_id"))
    
    # 3. Combine and Merge
    combined_df = inserts_df.unionByName(updates_df, allowMissingColumns=True)
    #display(combined_df)
    combined_df.createOrReplaceTempView("source_order_items_view")

    microBatchDF.sparkSession.sql(f"""
        MERGE INTO {target_table} AS target
        USING source_order_items_view AS source
        ON target.order_item_id = source.merge_key
        
        -- If we find the ID and status changed, expire the old one
        WHEN MATCHED AND source.product_id <> target.product_id AND source.quantity <> target.quantity THEN
            UPDATE SET 
            target.product_id = source.product_id,
            target.quantity = source.quantity,
            target.updated_timestamp = current_timestamp()
          
        -- If it's the 'NULL' half of the union, it will always be NOT MATCHED -> Insert
        WHEN NOT MATCHED THEN
          INSERT (
            order_item_sk, 
            order_item_id, 
            order_id, 
            product_id, 
            quantity, 
            unit_price, 
            updated_timestamp
          )
          VALUES (
            md5(source.order_item_id), 
            source.order_item_id, 
            source.order_id, 
            source.product_id, 
            source.quantity, 
            0.00, 
            current_timestamp()
          )
    """)

    # 3. Start the Stream
query = (spark.readStream
    .table("workspace.amazon_fullfilment_bronze.order_items")
    .filter("order_item_id IS NOT NULL") # Basic data quality
    .writeStream
    .foreachBatch(process_scd1_order_items)
    .option("checkpointLocation", f"{silver_volume_path}/checkpoints/silver_order_items_scd1")
    .trigger(availableNow=True) # Or .trigger(processingTime='1 minute') for 24/7
    .start())

In [0]:
# Generate Bin data

import uuid
import random
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, IntegerType, DoubleType, TimestampType


bin_schema = StructType([
    StructField("bin_id", StringType(), False),
    StructField("order_id", StringType(), True),
    StructField("item_count", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("current_weight", DoubleType(), True),
    StructField("employee_id", StringType(), True),
    StructField("updated_timestamp", TimestampType(), True)
])


def generate_bin_data(num_bins=50):
    bins = []
    for _ in range(num_bins):
        now = datetime.now()
        bins.append(Row(
            bin_id=str(uuid.uuid4()),
            order_id="",
            item_count=0,
            status="idle",
            current_weight=0.00,
            employee_id="",
            updated_timestamp=now
        
        ))
        
    return spark.createDataFrame(bins, schema=bin_schema)

# Generate and view
bin_df = generate_bin_data()
#display(bin_df)
#save to the source volume
(bin_df.write
.format("csv")
.option("header",True)
.option("delimiter",",")
.mode("append")
.save(f"{source_volume_path}/bin_events"))

#bin_df.write.format("delta").mode("overwrite").saveAsTable("workspace.amazon_fullfilment_silver.bins")

In [0]:
# Insert into Bronze layer bin_event table
 
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, DateType, IntegerType

# Define exactly what the data should look like
bin_schema = StructType([
    StructField("bin_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("item_count", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("current_weight", DoubleType(), True),
    StructField("employee_id", StringType(), True),
    StructField("updated_timestamp", TimestampType(), True)
])

# 1. Define your paths
bronze_bin_path = f"{bronze_volume_path}bin_events"

ingest_to_bronze("bin_events", bin_schema, f"{source_volume_path}bin_events", bronze_bin_path)

In [0]:
# process new records into the Silver layer bin table
from pyspark.sql import functions as F
from pyspark.sql.functions import current_timestamp, current_date

def process_scd2_bins(microBatchDF, batchId):
    # 1. Get a list of order_ids that already exist in Silver
    # This prevents duplicating brand new orders
    target_table = "workspace.amazon_fullfilment_silver.bins"
    
    # 2. Split the micro-batch logic
    # Part A: The 'Insert' half (Always NULL merge_key)
    inserts_df = microBatchDF.withColumn("merge_key", F.lit(None))
    
    # Part B: The 'Update' half (Only for IDs that already exist in Silver)
    # This prevents the 'double insert' on the first run
    existing_ids = microBatchDF.sparkSession.table(target_table) \
        .filter("is_active = true") \
        .select("bin_id")
    
    updates_df = microBatchDF.join(existing_ids, "bin_id", "inner") \
        .withColumn("merge_key", F.col("bin_id"))
    
    # 3. Combine and Merge
    combined_df = inserts_df.unionByName(updates_df, allowMissingColumns=True)
    combined_df.createOrReplaceTempView("source_bins_view")

    microBatchDF.sparkSession.sql(f"""
        MERGE INTO {target_table} AS target
        USING source_bins_view AS source
        ON target.bin_id = source.merge_key AND target.is_active = true
        
        -- If we find the ID and status changed, expire the old one
        WHEN MATCHED AND target.status <> source.status AND target.order_id <> source.order_id THEN
          UPDATE SET target.is_active = false, target.end_date = source.updated_timestamp, target.updated_timestamp = source.updated_timestamp
          
        -- If it's the 'NULL' half of the union, it will always be NOT MATCHED -> Insert
        WHEN NOT MATCHED THEN
          INSERT (bins_sk, bin_id, order_id, item_count, status, current_weight, is_active, start_date, end_date, updated_timestamp)
          VALUES (md5(concat(source.bin_id, cast(source.updated_timestamp as STRING))), 
                  source.bin_id, source.order_id, source.item_count, source.status, source.current_weight, 
                  true, source.updated_timestamp, NULL, source.updated_timestamp)
    """)

    # 3. Start the Stream
query = (spark.readStream
    .table("workspace.amazon_fullfilment_bronze.bin_events")
    .filter("status IS NOT NULL") # Basic data quality
    .writeStream
    .foreachBatch(process_scd2_bins)
    .option("checkpointLocation", f"{silver_volume_path}/checkpoints/silver_bin_scd2")
    .trigger(availableNow=True) # Or .trigger(processingTime='1 minute') for 24/7
    .start())